# Sprint 1 — BluaDiagnostics

## IA Conversacional para Check-up Digital

PoC acadêmica desenvolvida para o Challenge BluaDiagnostics utilizando:

- Prompt Engineering;
- memória conversacional;
- function calling simulado;
- integração com Ollama Cloud.

O notebook demonstra:

- system prompt clínico;
- guardrails;
- tools simuladas;
- fluxo de suporte ao beneficiário;
- contexto multi-turno.


## 1. Instalação das dependências

In [18]:
!pip install -q ollama ipython

## 2. Configuração do Ollama Cloud

No menu lateral esquerdo do Colab:

🔑 Secrets

Crie:

```text
OLLAMA_API_KEY
```

Cole sua chave do Ollama Cloud.


In [19]:
import os

try:
    from google.colab import userdata

    OLLAMA_API_KEY = userdata.get("OLLAMA_API_KEY")
    print("✅ Executando no Google Colab")

except:
    from dotenv import load_dotenv

    load_dotenv()
    OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY")

    print("✅ Executando localmente no VSCode")

✅ Executando no Google Colab


In [20]:
from ollama import Client

OLLAMA_API_KEY = userdata.get("OLLAMA_API_KEY")

if not OLLAMA_API_KEY:
    raise ValueError("Configure o Secret OLLAMA_API_KEY no Colab.")

client = Client(
    host="https://ollama.com",
    headers={
        "Authorization": "Bearer " + OLLAMA_API_KEY
    }
)

MODEL_NAME = "gpt-oss:120b"

print("✅ Ollama Cloud configurado com sucesso")


✅ Ollama Cloud configurado com sucesso


## 3. Teste de conexão

In [21]:
teste = client.chat(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Responda apenas OK"
        }
    ],
    stream=False
)

print("Resposta do modelo:")
print(teste["message"]["content"])


Resposta do modelo:
OK


## 4. System Prompt Clínico

O agente atua como assistente de apoio ao beneficiário da Care Plus.


In [22]:
system_prompt = """
Você é o BluaDiagnostics, um assistente conversacional da Care Plus.

OBJETIVO:
Auxiliar beneficiários em check-ups digitais e suporte pós-teleconsulta.

RESTRIÇÕES:
- Não forneça diagnóstico definitivo.
- Não prescreva medicamentos.
- Oriente procurar atendimento médico em situações críticas.
- Respeite LGPD e minimize coleta de dados sensíveis.

ESCALADA HUMANA:
Encaminhe imediatamente situações de:
- falta de ar intensa;
- dor no peito;
- desmaio;
- confusão mental;
- sinais neurológicos;
- sangramentos importantes.

FORMATO:
1. Resumo do caso
2. Pontos de atenção
3. Próxima ação
4. Aviso médico
"""

print(system_prompt)



Você é o BluaDiagnostics, um assistente conversacional da Care Plus.

OBJETIVO:
Auxiliar beneficiários em check-ups digitais e suporte pós-teleconsulta.

RESTRIÇÕES:
- Não forneça diagnóstico definitivo.
- Não prescreva medicamentos.
- Oriente procurar atendimento médico em situações críticas.
- Respeite LGPD e minimize coleta de dados sensíveis.

ESCALADA HUMANA:
Encaminhe imediatamente situações de:
- falta de ar intensa;
- dor no peito;
- desmaio;
- confusão mental;
- sinais neurológicos;
- sangramentos importantes.

FORMATO:
1. Resumo do caso
2. Pontos de atenção
3. Próxima ação
4. Aviso médico



## 5. Dados simulados do paciente

In [23]:
import json

paciente_mock = {
    "paciente_id": "PAC001",
    "nome": "Paciente Simulado",
    "idade": 42,
    "condicoes": ["hipertensão leve"],
    "medicamentos": ["losartana 50mg"],
    "alergias": ["dipirona"],
    "ultima_pressao": "135/85",
    "ultima_teleconsulta": "2026-05-10"
}

print(json.dumps(
    paciente_mock,
    indent=2,
    ensure_ascii=False
))


{
  "paciente_id": "PAC001",
  "nome": "Paciente Simulado",
  "idade": 42,
  "condicoes": [
    "hipertensão leve"
  ],
  "medicamentos": [
    "losartana 50mg"
  ],
  "alergias": [
    "dipirona"
  ],
  "ultima_pressao": "135/85",
  "ultima_teleconsulta": "2026-05-10"
}


## 6. Function Calling Simulado

In [24]:
def consultar_historico_paciente(paciente_id):
    print("🔧 Executando: consultar_historico_paciente")

    return paciente_mock


def verificar_interacoes_medicamentosas(medicamentos):
    print("🔧 Executando: verificar_interacoes_medicamentosas")

    if "ibuprofeno" in medicamentos:
        return {
            "risco": "moderado",
            "mensagem": "Possível atenção ao uso em paciente hipertenso."
        }

    return {
        "risco": "baixo",
        "mensagem": "Nenhuma interação relevante encontrada."
    }


def agendar_teleconsulta():
    print("🔧 Executando: agendar_teleconsulta")

    return {
        "status": "agendado",
        "data": "2026-05-20",
        "horario": "14:30"
    }

print("✅ Tools carregadas")


✅ Tools carregadas


## 7. Memória Conversacional

In [25]:
memoria = [
    {
        "role": "system",
        "content": system_prompt
    }
]

print("✅ Memória inicializada")


✅ Memória inicializada


## 8. Função principal do agente

In [26]:
from IPython.display import Markdown, display

def chamar_llm(mensagem_usuario):

    memoria.append({
        "role": "user",
        "content": mensagem_usuario
    })

    resposta = client.chat(
        model=MODEL_NAME,
        messages=memoria,
        options={
            "temperature": 0.3,
            "num_predict": 300
        },
        stream=False
    )

    texto = resposta["message"]["content"]

    memoria.append({
        "role": "assistant",
        "content": texto
    })

    display(Markdown(f"""
## 🤖 Resposta do BluaDiagnostics

{texto}
"""))

    return texto


## 9. Demonstração — Conversa Multi-turno

In [27]:
chamar_llm(
    "Olá, sou hipertenso e hoje acordei com dor de cabeça leve."
)



## 🤖 Resposta do BluaDiagnostics

**1. Resumo do caso**  
- Paciente relata ser hipertenso.  
- Hoje acordou com dor de cabeça leve.

**2. Pontos de atenção**  
- Verificar se a pressão arterial está dentro da faixa esperada (idealmente < 140/90 mmHg para hipertensos, conforme orientação do seu médico).  
- Avaliar se há outros sintomas associados: visão turva, tontura, náuseas, vômitos, sensação de pulsação na cabeça, aumento súbito da dor ou dor que não melhora com repouso.  
- Considerar fatores desencadeantes recentes: falta de sono, estresse, consumo excessivo de sal, álcool, cafeína ou interrupção da medicação antihipertensiva.  
- Monitorar a frequência e a intensidade da dor ao longo do dia.

**3. Próxima ação**  
1. **Meça a pressão arterial** (idealmente em braço esquerdo, em repouso, após 5 min de descanso).  
2. Anote o valor e a hora da medição


'**1. Resumo do caso**  \n- Paciente relata ser hipertenso.  \n- Hoje acordou com dor de cabeça leve.\n\n**2. Pontos de atenção**  \n- Verificar se a pressão arterial está dentro da faixa esperada (idealmente <\u202f140/90\u202fmmHg para hipertensos, conforme orientação do seu médico).  \n- Avaliar se há outros sintomas associados: visão turva, tontura, náuseas, vômitos, sensação de pulsação na cabeça, aumento súbito da dor ou dor que não melhora com repouso.  \n- Considerar fatores desencadeantes recentes: falta de sono, estresse, consumo excessivo de sal, álcool, cafeína ou interrupção da medicação antihipertensiva.  \n- Monitorar a frequência e a intensidade da dor ao longo do dia.\n\n**3. Próxima ação**  \n1. **Meça a pressão arterial** (idealmente em braço esquerdo, em repouso, após 5\u202fmin de descanso).  \n2. Anote o valor e a hora da medição'

In [28]:
chamar_llm(
    "Minha pressão ficou 135 por 85 e senti tontura ao levantar."
)



## 🤖 Resposta do BluaDiagnostics

**1. Resumo do caso**  
- Hipertensão conhecida.  
- Pressão arterial medida em repouso: **135 × 85 mmHg** (dentro da meta usual para hipertensos).  
- Relata **tontura ao levantar** (possível hipotensão ortostática).  
- Dor de cabeça leve continua presente.

**2. Pontos de atenção**  
- **Hipotensão ortostática**: queda súbita da pressão ao mudar de posição pode causar tontura, sensação de “cabeça leve” e piorar a dor de cabeça.  
- **Medicação antihipertensiva**: alguns fármacos (diuréticos, betabloqueadores, bloqueadores dos canais de cálcio) podem predispor a quedas de pressão ao ficar em pé.  
- **Desidratação ou ingestão insuficiente de sódio**: pode intensificar a queda de pressão.  
- **Sinais de alerta**: tontura que persiste após alguns minutos, desmaio,


'**1. Resumo do caso**  \n- Hipertensão conhecida.  \n- Pressão arterial medida em repouso: **135\u202f×\u202f85\u202fmmHg** (dentro da meta usual para hipertensos).  \n- Relata **tontura ao levantar** (possível hipotensão ortostática).  \n- Dor de cabeça leve continua presente.\n\n**2. Pontos de atenção**  \n- **Hipotensão ortostática**: queda súbita da pressão ao mudar de posição pode causar tontura, sensação de “cabeça leve” e piorar a dor de cabeça.  \n- **Medicação antihipertensiva**: alguns fármacos (diuréticos, betabloqueadores, bloqueadores dos canais de cálcio) podem predispor a quedas de pressão ao ficar em pé.  \n- **Desidratação ou ingestão insuficiente de sódio**: pode intensificar a queda de pressão.  \n- **Sinais de alerta**: tontura que persiste após alguns minutos, desmaio,'

## 10. Demonstração de memória

In [29]:
print("📌 Histórico armazenado:\n")

for item in memoria:
    print(item["role"].upper())
    print(item["content"][:300])
    print("-" * 50)


📌 Histórico armazenado:

SYSTEM

Você é o BluaDiagnostics, um assistente conversacional da Care Plus.

OBJETIVO:
Auxiliar beneficiários em check-ups digitais e suporte pós-teleconsulta.

RESTRIÇÕES:
- Não forneça diagnóstico definitivo.
- Não prescreva medicamentos.
- Oriente procurar atendimento médico em situações críticas.
- Re
--------------------------------------------------
USER
Olá, sou hipertenso e hoje acordei com dor de cabeça leve.
--------------------------------------------------
ASSISTANT
**1. Resumo do caso**  
- Paciente relata ser hipertenso.  
- Hoje acordou com dor de cabeça leve.

**2. Pontos de atenção**  
- Verificar se a pressão arterial está dentro da faixa esperada (idealmente < 140/90 mmHg para hipertensos, conforme orientação do seu médico).  
- Avaliar se há outros sint
--------------------------------------------------
USER
Minha pressão ficou 135 por 85 e senti tontura ao levantar.
--------------------------------------------------
ASSISTANT
**1. Resumo d

## 11. Demonstração de Tool Calling

In [30]:
historico = consultar_historico_paciente("PAC001")

print(json.dumps(
    historico,
    indent=2,
    ensure_ascii=False
))


🔧 Executando: consultar_historico_paciente
{
  "paciente_id": "PAC001",
  "nome": "Paciente Simulado",
  "idade": 42,
  "condicoes": [
    "hipertensão leve"
  ],
  "medicamentos": [
    "losartana 50mg"
  ],
  "alergias": [
    "dipirona"
  ],
  "ultima_pressao": "135/85",
  "ultima_teleconsulta": "2026-05-10"
}


In [31]:
interacoes = verificar_interacoes_medicamentosas(
    ["losartana", "ibuprofeno"]
)

print(json.dumps(
    interacoes,
    indent=2,
    ensure_ascii=False
))


🔧 Executando: verificar_interacoes_medicamentosas
{
  "risco": "moderado",
  "mensagem": "Possível atenção ao uso em paciente hipertenso."
}


In [32]:
teleconsulta = agendar_teleconsulta()

print(json.dumps(
    teleconsulta,
    indent=2,
    ensure_ascii=False
))


🔧 Executando: agendar_teleconsulta
{
  "status": "agendado",
  "data": "2026-05-20",
  "horario": "14:30"
}


## 12. Resposta contextualizada usando tools

In [ ]:
contexto = f'''
Histórico do paciente:
{historico}

Interações medicamentosas:
{interacoes}

Teleconsulta:
{teleconsulta}

Gere uma orientação segura para o paciente.
'''

chamar_llm(contexto)


# Conclusão

A PoC demonstrou:

- uso de Prompt Engineering;
- system prompt clínico;
- memória conversacional;
- function calling simulado;
- integração com Ollama Cloud;
- guardrails clínicos;
- fluxo Human-in-the-Loop.

O projeto atende os requisitos principais da Sprint 1 do Challenge BluaDiagnostics.
